# Federated duration + disposal-nature experiments (Tasks 1 & 4)

**How to run (12 h Kaggle limit):**
1. Upload the new `fl_judicial.py`, `run_queue.py`, `summarize_results.py` to your Kaggle dataset (replace the old `fl_judicial.py`), together with `judgments_labelled.csv`.
2. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
3. Session 1: keep `PLAN = "sweep"` and use **Save Version → Save & Run All (Commit)**. It runs in the background (browser can be closed) and stops itself before the 12 h limit.
4. Session 2: open the notebook, **Add Input → Your Work → this notebook's output (the version from session 1)**, set `PLAN = "confirm"`, and Save & Run All again.
   - If session 1 did not finish its queue, run `"sweep"` once more first: finished runs are skipped and an interrupted FL run resumes from its last round.
5. Results: `runs/all_results.csv`, `comparison.csv`, `per_court.png` in the notebook output.

Everything is selected on **validation** loss; test numbers are only reported.

In [ ]:
!pip install -q peft
!pip uninstall -y -q torchao
import os, glob, shutil, subprocess, sys, time, torch
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None
DATA = find("judgments_labelled.csv")
for f in ["fl_judicial.py", "run_queue.py", "summarize_results.py"]:
    shutil.copy(find(f), f"/kaggle/working/{f}")
os.chdir("/kaggle/working")
N_GPU = torch.cuda.device_count()
print("data:", DATA, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert "init_from" in open("fl_judicial.py").read(), "old fl_judicial.py in the dataset: upload the new one" 

In [ ]:
# Resume: bring back results/checkpoints from a previous session's output (if attached as input)
prev = [p for p in glob.glob("/kaggle/input/**/runs/all_results.csv", recursive=True)]
for p in prev:
    src = os.path.dirname(p)
    shutil.copytree(src, "/kaggle/working/runs", dirs_exist_ok=True)
    print("restored previous runs from", src)
print("runs so far:", len(os.listdir("runs")) if os.path.exists("runs") else 0)

In [ ]:
PLAN = "sweep"          # session 1: "sweep"   |   session 2: "confirm"
BUDGET_HOURS = 10.8     # each queue stops launching new runs after this; Kaggle hard limit is 12 h
ENCODERS = ["roberta-base", "law-ai/InLegalBERT"]

def launch(gpu, extra, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, "-u", "run_queue.py", "--data", DATA, "--plan", PLAN, "--budget_hours", str(BUDGET_HOURS), *extra]
    return subprocess.Popen(cmd, env=env, stdout=open(log, "a"), stderr=subprocess.STDOUT)

t0 = time.time(); procs = []
if N_GPU >= 2:
    if PLAN == "sweep":
        procs = [launch(i, ["--encoder", e], f"queue_gpu{i}.log") for i, e in enumerate(ENCODERS[:2])]
    else:
        procs = [launch(i, ["--shard", str(i), "--n_shards", "2"], f"queue_gpu{i}.log") for i in range(2)]
else:   # one GPU: queues run one after another (remaining work continues next session)
    jobs = [["--encoder", e] for e in ENCODERS] if PLAN == "sweep" else [["--shard", "0", "--n_shards", "1"]]
    def serial():
        for j in jobs:
            subprocess.run([sys.executable, "-u", "run_queue.py", "--data", DATA, "--plan", PLAN,
                            "--budget_hours", str(BUDGET_HOURS - (time.time() - t0) / 3600), *j],
                           stdout=open("queue_gpu0.log", "a"), stderr=subprocess.STDOUT)
    import threading; th = threading.Thread(target=serial); th.start()
    class P:  # minimal Popen-like wrapper
        def poll(self): return None if th.is_alive() else 0
    procs = [P()]
print("launched", len(procs), "queue(s)")

In [ ]:
# Monitor until the queues finish (prints progress every 10 minutes)
import re
while any(p.poll() is None for p in procs):
    time.sleep(600)
    print(f"\n===== {(time.time()-t0)/3600:.2f} h =====")
    for log in sorted(glob.glob("queue_gpu*.log")):
        lines = [l for l in open(log).read().splitlines() if re.search(r"start|mean:|chosen|early stop|best round|Error|skipped", l)]
        print(log, "\n  " + "\n  ".join(lines[-6:]))
print("all queues finished")
for log in sorted(glob.glob("queue_gpu*.log")):
    print(open(log).read()[-3000:])

In [ ]:
!python summarize_results.py runs/all_results.csv

In [ ]:
# Per-court view of seed-0 runs at the chosen setting (best FedAvg on validation)
import pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv("runs/all_results.csv"); r = r[(r.seed == 0) & (r.max_len == 256)]
fa = r[r.method == "fedavg"].groupby(["model", "lr", "trunc"]).sel_val_loss.mean()
for (enc, lr, tr) in fa.sort_values().index[:2]:
    s = r[(r.model == enc) & (r.lr == lr) & (r.trunc == tr)]
    s = s.assign(label=s.method + " | " + s.run.str.extract(r"(mu[\d.]+|beta-?[\d.]+|slr[\d.e-]+|m0\.5)", expand=False).fillna(""))
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    for a, m in zip(ax, ["MAE_log", "disp_F1"]):
        s.pivot_table(index="court", columns="label", values=m).plot.bar(ax=a, legend=(m == "disp_F1"))
        a.set_title(f"{m} per court — {enc}, lr={lr}, {tr}"); a.set_xlabel("")
    ax[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(f"per_court_{enc.split('/')[-1]}.png", dpi=130); plt.show()